In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


summary_path = Path("/home/akapociu/ift/interactiondynamics/results/interaction_predictions_misfires/summary.jsonl")


def extract_model_combo(run_name: str) -> str:
    """
    Pulls out agg=... and update=... from a long run name and returns a short label.
    """
    run_name = str(run_name)

    agg_match = re.search(r"agg=([^|]+)", run_name)
    upd_match = re.search(r"update=([^|]+)", run_name)

    agg = agg_match.group(1) if agg_match else "unknown_agg"
    upd = upd_match.group(1) if upd_match else "unknown_update"

    pretty_agg = {
        "sum": "Sum",
        "deepsets": "DeepSets",
        "settransformer": "SetTransformer",
        "hopfield": "Hopfield",
        "ift": "IFT",
    }.get(agg, agg)

    pretty_upd = {
        "tgn_gru": "TGN-GRU",
        "ift_update": "IFT Update",
        "hopfield_update": "Hopfield Update",
        "hnn": "HNN",
        "lnn": "LNN",
    }.get(upd, upd)

    return f"{pretty_agg} + {pretty_upd}"


rows = []
with open(summary_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        row = json.loads(line)
        recovery_val = row.get("recovery_val")
        if not recovery_val:
            continue

        val_hidden_mrr = recovery_val.get("mrr")
        if val_hidden_mrr is None:
            continue

        run_name = row.get("run", "unknown")

        rows.append(
            {
                "dataset": row.get("dataset", "unknown"),
                "run": run_name,
                "model_combo": extract_model_combo(run_name),
                "seed": row.get("seed"),
                "val_hidden_mrr": float(val_hidden_mrr),
                "val_hidden_hits10": recovery_val.get("hits@10"),
                "val_removed": recovery_val.get("removed_events"),
            }
        )

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError("No recovery_val.mrr entries found in summary.jsonl")

avg_df = (
    df.groupby(["dataset", "model_combo"], as_index=False)
      .agg(
          avg_val_hidden_mrr=("val_hidden_mrr", "mean"),
          std_val_hidden_mrr=("val_hidden_mrr", "std"),
          n_seeds=("seed", "count"),
      )
)

for dataset_name, subdf in avg_df.groupby("dataset"):
    plot_df = subdf.sort_values("avg_val_hidden_mrr", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(10, max(4, 0.5 * len(plot_df))))
    plt.barh(
        plot_df["model_combo"],
        plot_df["avg_val_hidden_mrr"],
        xerr=plot_df["std_val_hidden_mrr"].fillna(0.0),
        capsize=4,
    )
    plt.gca().invert_yaxis()
    plt.xlabel("Average Validation Hidden MRR")
    plt.ylabel("Model Combo")
    plt.title(f"{dataset_name}: Average Validation Hidden MRR Across Seeds")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()